# 3. Random Sampling

NumPy has two random APIs:

1. **The `Generator` API** (`np.random.default_rng()`). This is what you should use in new code. It's faster, has better statistical properties, and doesn't rely on global state.
2. **The legacy API** (`np.random.rand`, `np.random.seed`, ...). It still works and you'll see it all over older code and tutorials, so it's covered at the end.

In [1]:
import numpy as np

## Creating a generator

Pass a seed if you want the same numbers every time you run the notebook. Two generators with the same seed produce identical streams, and they don't interfere with each other.

In [2]:
rng = np.random.default_rng(123)
rng.random(3)

array([0.68235186, 0.05382102, 0.22035987])

In [3]:
a = np.random.default_rng(7)
b = np.random.default_rng(7)
a.random(3), b.random(3)

(array([0.62509547, 0.8972138 , 0.77568569]),
 array([0.62509547, 0.8972138 , 0.77568569]))

## Uniform floats on [0, 1)

In [4]:
print(rng.random())          # a single float
print(rng.random(3))         # 1-D
print(rng.random((2, 3)))    # 2-D, note the shape is a tuple

0.1843718106986697
[0.1759059  0.81209451 0.923345  ]
[[0.2765744  0.81975456 0.88989269]
 [0.51297046 0.2449646  0.8242416 ]]


For a different range use `uniform(low, high, size)`:

In [5]:
rng.uniform(-5, 5, size=4)

array([-2.86237037,  2.41467052,  1.29940205,  4.27407259])

## Integers

`integers(low, high, size)` samples from `[low, high)`. Pass `endpoint=True` if you want `high` included.

In [6]:
print(rng.integers(1, 10, size=5))
print(rng.integers(1, 6, size=10, endpoint=True))   # roll a die ten times

[7 3 8 8 2]
[4 5 2 2 1 1 3 1 4 3]


## Common distributions

In [7]:
print(rng.standard_normal((2, 3)))                 # mean 0, std 1
print(rng.normal(loc=170, scale=10, size=5))       # heights in cm, say
print(rng.binomial(n=10, p=0.5, size=5))           # heads in 10 coin flips
print(rng.poisson(lam=3, size=5))
print(rng.exponential(scale=2.0, size=3))

[[-0.14597789  1.28190223  1.07403062]
 [ 0.39262084  0.00511431 -0.36176687]]
[157.69767805 182.26229293 148.27956113 166.29852654 171.6438007 ]
[6 7 4 5 3]
[3 1 0 2 1]
[1.35098514 1.53062042 1.98582649]


A quick check that the normal samples look right:

In [8]:
x = rng.normal(loc=5, scale=2, size=100_000)
print(round(x.mean(), 2), round(x.std(), 2))

5.0 2.0


## Picking from an existing array

`choice(a, size, replace=True, p=None)`

- `a` can be an array or an int (`n` means `np.arange(n)`)
- `replace=False` samples without replacement
- `p` gives a probability for each element (must sum to 1)

In [9]:
print(rng.choice(['red', 'green', 'blue'], size=8))
print(rng.choice(3, size=10, p=[0.1, 0.8, 0.1]))
print(rng.choice(10, size=5, replace=False))

['red' 'blue' 'red' 'red' 'red' 'blue' 'green' 'green']
[2 1 1 1 1 1 1 1 1 1]
[5 2 4 8 1]


## Shuffling

- `shuffle` changes the array **in place** and returns `None`
- `permutation` returns a shuffled **copy**
- `permuted` shuffles each row (or column) independently

For 2-D arrays, `shuffle` and `permutation` only reorder along the first axis, so rows stay intact.

In [10]:
arr = np.arange(10)
rng.shuffle(arr)
arr

array([1, 9, 2, 8, 7, 3, 4, 6, 0, 5])

In [11]:
arr = np.arange(10)
print('original:', arr)
print('permuted:', rng.permutation(arr))
print('original is unchanged:', arr)

original: [0 1 2 3 4 5 6 7 8 9]
permuted: [8 2 4 0 1 7 6 3 9 5]
original is unchanged: [0 1 2 3 4 5 6 7 8 9]


In [12]:
grid = np.arange(12).reshape(3, 4)
print(rng.permutation(grid))           # rows move as whole units
print(rng.permuted(grid, axis=1))      # each row gets shuffled separately

[[ 4  5  6  7]
 [ 8  9 10 11]
 [ 0  1  2  3]]
[[ 3  1  0  2]
 [ 4  7  6  5]
 [ 9 10  8 11]]


### Example: a train/test split

A typical use of `permutation`, shuffling indices and slicing them:

In [13]:
X = np.arange(20).reshape(10, 2)
y = np.arange(10)

idx = rng.permutation(len(X))
cut = int(0.8 * len(X))
train_idx, test_idx = idx[:cut], idx[cut:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(X_train.shape, X_test.shape)
print(y_test)

(8, 2) (2, 2)
[0 5]


## The legacy API

You'll run into these in older code. They all draw from one hidden global generator, which is why `np.random.seed` affects everything, including code in libraries you call. That makes it easy to get results that aren't actually reproducible.

| Legacy | Generator equivalent |
|---|---|
| `np.random.seed(s)` | `rng = np.random.default_rng(s)` |
| `np.random.rand(3, 3)` | `rng.random((3, 3))` |
| `np.random.randn(3, 3)` | `rng.standard_normal((3, 3))` |
| `np.random.randint(lo, hi, n)` | `rng.integers(lo, hi, n)` |
| `np.random.random_sample(n)` | `rng.random(n)` |
| `np.random.choice(...)` | `rng.choice(...)` |
| `np.random.shuffle(a)` | `rng.shuffle(a)` |
| `np.random.permutation(a)` | `rng.permutation(a)` |

If you need an isolated legacy generator, `np.random.RandomState(seed)` gives you one.

In [14]:
np.random.seed(123)
print(np.random.rand(3))
print(np.random.randint(1, 10, 3))

rs = np.random.RandomState(321)    # independent from the global seed
print(rs.rand(3))

[0.69646919 0.28613933 0.22685145]
[7 2 4]
[0.88594794 0.07791236 0.97964616]
